### 指定日期重跑對比
- 從 `save_path/{check_date}/` 讀歸檔 zip，重跑與 `daily_data_update` 相同的處理流程
- 結果 `final_df` 與 DuckDB `daily_data` 同日資料比對差異（**不寫入 DB、不歸檔、不改 halt_calendar**）
- zip **不解壓落地**：`read_tej_zip` 只讀進記憶體
- **模組**沿用 `rebuild_data/`（schema / parse）

In [2]:
%reset -f

In [3]:
import sys
import os
import zipfile
import importlib

import polars as pl

# Python 模組：rebuild_data（與全量重建共用）
modules_root = "/home/newt/quant_project/rebuild_data"
if modules_root not in sys.path:
    sys.path.insert(0, modules_root)

# 每日增量資料根目錄（獨立，勿與 rebuild_root 混用）
update_root = "/mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily"
save_path = os.path.join(update_root, "daily_save")  # 歸檔 zip：daily_save/YYYY-MM-DD/

# ★ 指定要重跑對比的日期（對應 save_path 子目錄名）
check_date = "2026-07-30"
zip_dir = os.path.join(save_path, check_date)  # 本 notebook 的讀取來源

if not os.path.isdir(zip_dir):
    raise FileNotFoundError(
        f"找不到日期目錄：{zip_dir}\n"
        f"可用日期：{sorted(d for d in os.listdir(save_path) if os.path.isdir(os.path.join(save_path, d)))}"
    )

_zips = sorted(n for n in os.listdir(zip_dir) if n.endswith(".zip"))
print("modules_root:", modules_root)
print("update_root :", update_root)
print("save_path   :", save_path)
print("check_date  :", check_date)
print("zip_dir     :", zip_dir, f"({len(_zips)} zips)")
for n in _zips:
    print(f"  - {n}")

modules_root: /home/newt/quant_project/rebuild_data
update_root : /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily
save_path   : /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily/daily_save
check_date  : 2026-07-30
zip_dir     : /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily/daily_save/2026-07-30 (11 zips)
  - 三大法人買賣超 - 2026-08-04 203413.zip
  - 借券餘額 - 2026-08-04 203455.zip
  - 基本資料 - 2026-08-04 203446.zip
  - 暫停交易記錄 - 2026-08-04 203438.zip
  - 未調整股價(日) - 2026-08-04 203504.zip
  - 現金股息 - 2026-08-04 203511.zip
  - 當日沖銷交易統計資訊 - 2026-08-04 203523.zip
  - 籌碼分佈(2008年起) - 2026-08-04 203541.zip
  - 籌碼動向＆報酬(2008年起) - 2026-08-04 203421.zip
  - 融資融券 - 2026-08-04 203429.zip
  - 資本形成 - 2026-08-04 203531.zip


In [4]:
from parse import read_tej_txt, read_tej_zip, find_tej_zip


def reload_modules(module_list):
    for mod in module_list:
        importlib.reload(mod)


def extract_zips_inplace(zip_dir: str) -> list[str]:
    """備援：把 zip 內 .txt 解到 zip 同目錄（不另開 daily_work）。

    日常請用 read_tej_zip（記憶體）；只有外部工具一定要實體 txt 時才呼叫。
    """
    stems = []
    for name in sorted(os.listdir(zip_dir)):
        if not name.endswith(".zip"):
            continue
        zpath = os.path.join(zip_dir, name)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(zip_dir)
        stems.append(name[:-4])
        print(f"extracted inplace: {name} → {zip_dir}")
    return stems


## 資料載入（從 `save_path/{check_date}` 重跑）
一律 `read_tej_zip(zip_dir, 名稱前綴, …)`，不解壓到 SSD。
`calendar` / `company_info` 若當日 zip 沒有則 skip。
本 notebook **不更新** `halt_calendar.json`（只用既有累積檔套用）。

### Trading_Halt — 暫停交易


In [5]:
import Trading_Halt
reload_modules([Trading_Halt])
halt_calendar_path = "/home/newt/quant_project/src/manage_database/TW_stock/halt_calendar.json"

trading_halt_df = read_tej_zip(
    zip_dir,
    "暫停交易記錄",
    Trading_Halt.TRADING_HALT_SCHEMA,
).select(Trading_Halt.keep_col)
print("trading_halt_df:", trading_halt_df.shape)

if trading_halt_df.height == 0:
    print("當日無暫停交易紀錄，halt_calendar 不更新")
else:
    n_new = Trading_Halt.update_halt_calendar(trading_halt_df, halt_calendar_path)
    print(f"已寫入 halt_calendar：新增 {n_new} 筆 → {halt_calendar_path}")
    display(trading_halt_df)


trading_halt_df: (1, 4)
已寫入 halt_calendar：新增 0 筆 → /home/newt/quant_project/src/manage_database/TW_stock/halt_calendar.json


symbol,start_halt,resume_trading_date,reason
str,date,date,str
"""5904""",2026-07-30,2026-08-10,"""面額變更，10元=>1元"""


### OHLCV（Daily_Unadjusted_Price）— 日收盤價（未復權）


In [6]:
import OHLCV
from OHLCV import OHLCV_SCHEMA, encode_ohlcv_codes

OHLCV_df = read_tej_zip(
    zip_dir,
    "未調整股價(日)",
    OHLCV_SCHEMA,
).select(OHLCV.keep_col)
print("OHLCV_df:", OHLCV_df.shape)
OHLCV_df = OHLCV_df.filter(OHLCV_df["market"].is_in(["TSE", "OTC"]))
OHLCV_df = encode_ohlcv_codes(OHLCV_df)

# 全額交割 → untradable=1；其餘 0（halt 稍後覆寫）
OHLCV_df = OHLCV_df.with_columns(
    [
        (pl.col("full_delivery") == 1)
        .fill_null(False)
        .cast(pl.Int8)
        .alias("untradable"),
        pl.lit(1.0).alias("adj_factor"),
        pl.lit(0).cast(pl.Int8).alias("avoid_early"),
    ]
)
OHLCV_df.head()


OHLCV_df: (2332, 22)


symbol,date,open,high,low,close,volume,amount,return_day,turnover,market_value_million,log_return_day,numtrades,limit_status,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market,untradable,adj_factor,avoid_early
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,i8,i8,i8,i8,i8,f64,i8
"""1101""",2026-07-30,24.45,24.5,24.0,24.05,26765.0,647825.0,-1.232,0.3558,180933.0,-1.2397,9018.0,0,2.0534,24.05,26.45,21.65,0,0,0,1,0,1.0,0
"""1102""",2026-07-30,33.0,33.4,32.75,33.2,11167.0,369482.0,1.2195,0.3149,117746.0,1.2121,4824.0,0,1.9817,33.2,36.5,29.9,0,0,0,1,0,1.0,0
"""1103""",2026-07-30,13.35,13.35,13.15,13.15,528.0,6982.0,-0.3788,0.0634,10952.0,-0.3795,273.0,0,1.5152,13.15,14.45,11.85,0,0,0,1,0,1.0,0
"""1104""",2026-07-30,26.8,27.25,26.6,27.2,1158.0,31336.0,1.3035,0.1686,18678.0,1.2951,713.0,0,2.4209,27.2,29.9,24.5,0,0,0,1,0,1.0,0
"""1108""",2026-07-30,12.0,12.05,11.85,11.85,355.0,4261.0,-0.8368,0.0877,4796.0,-0.8403,180.0,0,1.6736,11.85,13.0,10.7,0,0,0,1,0,1.0,0


In [7]:
unique_symbol = OHLCV_df["symbol"].unique().to_list()


### 套用 halt_calendar
- date ∈ `[start_halt, resume_trading_date)` → `untradable=1`
- reason 含「減資」→ `avoid_early=T`，標在 **start_halt 前、本批 OHLCV 最後一個交易日** 的 `avoid_early=1`
- calendar 裡有、但本批 OHLCV 沒有的 symbol 會印出提示（例如停牌當天已無報價）


In [8]:
import Trading_Halt

reload_modules([Trading_Halt])
halt_calendar_path = "/home/newt/quant_project/src/manage_database/TW_stock/halt_calendar.json"

OHLCV_df = Trading_Halt.apply_halt_calendar_to_ohlcv(OHLCV_df, halt_calendar_path)
print(
    "after halt_calendar: untradable=1 →",
    OHLCV_df.filter(pl.col("untradable") == 1).height,
    "| avoid_early=1 →",
    OHLCV_df.filter(pl.col("avoid_early") == 1).height,
)
display(OHLCV_df.filter((pl.col("untradable") == 1) | (pl.col("avoid_early") == 1)).head())

n_pruned = Trading_Halt.prune_halt_calendar(OHLCV_df, halt_calendar_path)
print(f"prune_halt_calendar: removed {n_pruned}")

after halt_calendar: untradable=1 → 32 | avoid_early=1 → 0


symbol,date,open,high,low,close,volume,amount,return_day,turnover,market_value_million,log_return_day,numtrades,limit_status,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market,untradable,adj_factor,avoid_early
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,i8,i8,i8,i8,i8,f64,i8
"""1213""",2026-07-30,7.11,7.11,7.1,7.11,5.0,35.0,-1.7956,0.0088,403.0,-1.8119,5.0,0,0.1381,7.11,7.82,6.4,0,0,1,1,1,1.0,0
"""1459""",2026-07-30,11.85,11.85,11.85,11.85,0.0,0.0,0.0,0.0,4250.0,0.0,0.0,0,0.0,11.85,null,null,0,0,0,1,1,1.0,0
"""1538""",2026-07-30,10.2,10.2,10.2,10.2,1.0,10.0,2.0,0.0031,329.0,1.9803,2.0,0,0.0,10.2,11.2,9.18,0,0,1,1,1,1.0,0
"""1589""",2026-07-30,5.54,5.54,5.54,5.54,0.0,0.0,0.0,0.0,848.0,0.0,0.0,0,0.0,5.54,null,null,0,0,0,1,1,1.0,0
"""2314""",2026-07-30,11.3,11.55,11.3,11.35,131.0,1501.0,0.4425,0.0695,2139.0,0.4415,129.0,0,2.2124,11.35,12.45,10.25,0,0,1,1,1,1.0,0


prune_halt_calendar: removed 0


### Cash_Dividends — 現金股息
對齊 OHLCV `(symbol, date)`、同日去重；保留 `ex_div_ref_price` 供合併時與權利重疊檢查。


In [9]:
import Cash_Dividends
reload_modules([Cash_Dividends])
cash_dividend_df = Cash_Dividends.prepare_for_merge(
    Cash_Dividends.process_cash_dividends(
        read_tej_zip(
            zip_dir,
            "現金股息",
            Cash_Dividends.CASH_DIVIDENDS_SCHEMA,
        )
    ),
    OHLCV_df,
)
has_cash_dividend = cash_dividend_df.height > 0
print("has_cash_dividend:", has_cash_dividend, "| shape:", cash_dividend_df.shape)
if has_cash_dividend:
    display(cash_dividend_df.head())


prev_close / ex_div_ref_price：無空值
cash dividend 無 OHLCV pair：drop 6 筆
has_cash_dividend: True | shape: (32, 4)


symbol,date,adj_factor,ex_div_ref_price
str,date,f64,f64
"""1216""",2026-07-30,1.039894,75.2
"""1231""",2026-07-30,1.118682,80.72
"""1514""",2026-07-30,1.041139,94.8
"""2233""",2026-07-30,1.022039,181.5
"""2345""",2026-07-30,1.007916,1895.0


### Corporate_Actions — 資本形成（除權／增減資等）


In [10]:
import Corporate_Actions
from Corporate_Actions import CORPORATE_ACTIONS_SCHEMA

ex_rights_df = read_tej_zip(
    zip_dir,
    "資本形成",
    CORPORATE_ACTIONS_SCHEMA,
)
print("ex_rights_df:", ex_rights_df.shape)
ex_rights_df.head()

ex_rights_df: (11, 74)


symbol,date,cash_capital_increase_thousand,retained_earnings_increase_thousand,capital_reserve_increase_thousand,capital_reduction_thousand,capital_reduction_cash_per_share,capital_reduction_rate,par_value_change_shares_thousand,par_value_change_ratio,cash_subscription_rate,cash_subscription_price,retained_earnings_stock_rate,capital_reserve_stock_rate,ex_rights_ref_price,prev_close,stock_dividend_yn,private_placement_yn,total_capital_thousand,par_value,currency,total_shares_thousand,distributable_shares_thousand,employee_bonus_shares_thousand,cb_conversion_shares_thousand,pref_conversion_shares_thousand,share_swap_thousand,treasury_cancel_thousand,rsa_cancel_thousand,merger_shares_thousand,transfer_shares_thousand,eso_shares_thousand,rsa_shares_thousand,other_shares_thousand,employee_stock_rate,cb_conversion_price,employee_subscription_rate,public_underwriting_rate,book_building_rate,gdr_rate,specific_rate,capital_change_announce_date,shareholder_meeting_date,sfc_approval_date,book_closure_start,book_closure_end,shareholder_payment_start,shareholder_payment_end,cash_call_start,cash_call_end,payment_certificate_announce_date,payment_certificate_issue_date,payment_certificate_list_date,cash_increase_stock_announce_date,cash_increase_stock_issue_date,cash_increase_stock_list_date,rights_certificate_announce_date,rights_certificate_issue_date,rights_certificate_list_date,employee_bonus_announce_date,employee_bonus_issue_date,employee_bonus_list_date,stock_dividend_announce_date,stock_dividend_issue_date,stock_dividend_list_date,last_transfer_date,margin_buy_suspend_start,margin_suspend_end,margin_sell_suspend_start,short_cover_deadline,cash_increase_failed,capital_increase_fail_reason,other_note,capital_reduction_refund_date
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,str,str,str,date
"""1231""",2026-07-30,0.0,29845.0,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,10.0,0.0,80.72,90.3,"""Y""","""N""",3.282999e6,10.0,"""NTD""",328300.0,298454.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-07-14,2026-06-24,2026-07-13,2026-08-03,2026-08-07,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-07-31,null,2026-07-29,2026-07-24,2026-07-24,"""N""",null,null,null
"""1342""",2026-07-30,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",785009.0,10.0,"""NTD""",78501.0,78001.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,500.0,0.0,null,0.0,null,null,null,null,null,2026-07-30,null,2026-06-30,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,"""限制員工權利新股""",null
"""1514""",2026-07-30,0.0,5462.0,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,2.0,0.0,94.8,98.7,"""Y""","""N""",2.785706e6,10.0,"""NTD""",278571.0,273108.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-07-03,2026-06-11,2026-06-22,2026-08-01,2026-08-05,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-07-31,null,2026-07-29,2026-07-24,2026-07-24,"""N""",null,null,null
"""2342""",2026-07-30,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",1.573352e6,10.0,"""NTD""",157335.0,157475.0,0.0,0.0,0.0,0.0,0.0,140.0,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,null,null,null,2026-07-30,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,"""限制員工權利新股註銷""",null
"""2439""",2026-07-30,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",2.486413e6,10.0,"""NTD""",248641.0,253645.0,0.0,0.0,0.0,0.0,5000.0,4.0,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,null,null

#### 配股（盈餘／公積增資）
無配股時 `has_stock_dividend=False`，後續合併可跳過。


In [11]:
reload_modules([Corporate_Actions])
ex_rights_adj_factor_df, has_stock_dividend = Corporate_Actions.process_stock_dividend(ex_rights_df)
print("has_stock_dividend:", has_stock_dividend)
if has_stock_dividend:
    print("ex_rights_adj_factor_df:", ex_rights_adj_factor_df.shape)
    display(ex_rights_adj_factor_df.head())
else:
    print("無配股，跳過後續配股合併")

配股 adj_factor 唯一 OK，筆數: 3
has_stock_dividend: True
ex_rights_adj_factor_df: (3, 4)


symbol,date,adj_factor,ref_price
str,date,f64,f64
"""1231""",2026-07-30,1.118682,80.72
"""1514""",2026-07-30,1.041139,94.8
"""7892""",2026-07-30,1.280594,536.47


#### 減資
停牌對照用 `halt_calendar.json`（累積）。無減資時 `has_capital_reduction=False`，後續合併可跳過。


In [12]:
reload_modules([Corporate_Actions, Trading_Halt])
# 停牌對照用累積 halt_calendar，勿只用當日 zip
halt_for_adj = Trading_Halt.load_halt_calendar(halt_calendar_path)
print("halt_for_adj (calendar):", halt_for_adj.shape)

capital_reduction_only_df, capital_reduction_halt_df, has_capital_reduction = (
    Corporate_Actions.process_capital_reduction(
        ex_rights_df,
        halt_for_adj,
        exclude_pairs=ex_rights_adj_factor_df.select(["symbol", "date"]),
    )
)
print("has_capital_reduction:", has_capital_reduction)
if has_capital_reduction:
    print("capital_reduction_only_df:", capital_reduction_only_df.shape)
    print("capital_reduction_halt_df:", capital_reduction_halt_df.shape)
    display(capital_reduction_only_df.head())
    display(capital_reduction_halt_df.head())
else:
    print("無減資，跳過後續減資合併")


halt_for_adj (calendar): (5, 4)
capital_reduction_thousand 有值 (非空且大於0) 的資料筆數: 0
其中 ex_rights_ref_price 和 prev_close 皆不為空的筆數: 0
無減資資料，跳過後續減資流程
has_capital_reduction: False
無減資，跳過後續減資合併


#### 面額變更
停牌對照用 `halt_calendar.json`（累積）。無面額變更時 `has_par_value_change=False`，後續合併可跳過。


In [13]:
reload_modules([Corporate_Actions, Trading_Halt])
# 停牌對照用累積 halt_calendar，勿只用當日 zip
halt_for_adj = Trading_Halt.load_halt_calendar(halt_calendar_path)
print("halt_for_adj (calendar):", halt_for_adj.shape)

par_value_change_path = (
    "/home/newt/quant_project/src/manage_database/TW_stock/par_value_change.json"
)

par_value_change_only_df, par_value_change_halt_df, has_par_value_change = (
    Corporate_Actions.process_par_value_change(
        ex_rights_df,
        halt_for_adj,
        exclude_pairs=ex_rights_adj_factor_df.select(["symbol", "date"]),
    )
)

# 有恢復日的面額變更：寫入 JSON；再併入 pending（含尚未到期的歷史列）
n_par_new = Corporate_Actions.update_par_value_change_calendar(
    par_value_change_only_df, par_value_change_path
)
print(f"par_value_change.json 新增 {n_par_new} 筆 → {par_value_change_path}")
pending_par = Corporate_Actions.load_par_value_change_calendar(par_value_change_path)
if pending_par.height:
    par_value_change_only_df = (
        pl.concat([par_value_change_only_df, pending_par], how="diagonal_relaxed")
        .unique(subset=["symbol", "date"], keep="first")
        .sort(["symbol", "date"])
    )

# 合併只用本批 OHLCV 對得到的 date；未來恢復日留在 JSON
_ohlcv_dates = OHLCV_df["date"].unique().to_list()
par_value_change_only_df = par_value_change_only_df.filter(
    pl.col("date").is_in(_ohlcv_dates)
)
has_par_value_change = par_value_change_only_df.height > 0

# 恢復當天（及之前）從 JSON 刪掉；合併仍用上面記憶體列
ref_date = OHLCV_df["date"].max()
n_par_pruned = Corporate_Actions.prune_par_value_change_calendar(
    ref_date, par_value_change_path
)
print(f"prune_par_value_change: removed {n_par_pruned} (ref={ref_date})")

print("has_par_value_change:", has_par_value_change)
if has_par_value_change:
    print("par_value_change_only_df:", par_value_change_only_df.shape)
    print("par_value_change_halt_df:", par_value_change_halt_df.shape)
    display(par_value_change_only_df.head())
    display(par_value_change_halt_df.head())
else:
    print("無面額變更，跳過後續面額變更合併")




halt_for_adj (calendar): (5, 4)
面額變更總筆數：1；落在停牌區間：1；有恢復日：1；有 adj_factor：1
面額變更_only：1 筆掛恢復日、0 筆掛事件日、合計 1
面額變更_only 唯一 OK，筆數: 1
par_value_change.json 新增 1 筆 → /home/newt/quant_project/src/manage_database/TW_stock/par_value_change.json
prune_par_value_change: removed 0 (ref=2026-07-30)
has_par_value_change: False
無面額變更，跳過後續面額變更合併


### institutional_return — 籌碼動向＆報酬


In [14]:
import institutional_return

reload_modules([institutional_return])
institutional_return_df = institutional_return.process_institutional_return(
    read_tej_zip(
        zip_dir,
        "籌碼動向＆報酬(2008年起)",
        institutional_return.INSTITUTIONAL_RETURN_SCHEMA,
    )
)
print("institutional_return_df:", institutional_return_df.shape)
institutional_return_df = institutional_return_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", institutional_return_df.shape)
institutional_return_df.head()

institutional_return_df: (2332, 31)
after symbol filter: (1944, 31)


symbol,date,foreign_net_buy_lot,it_net_buy_lot,inst_net_buy_lot,margin_net_buy_lot,cash_net_buy_lot,foreign_net_buy_days,it_net_buy_days,dealer_net_buy_days,inst_net_buy_days,margin_net_buy_days,cash_net_buy_days,foreign_streak_return,it_streak_return,dealer_streak_return,inst_streak_return,margin_streak_return,cash_streak_return,foreign_streak_cum_lot,it_streak_cum_lot,dealer_streak_cum_lot,inst_streak_cum_lot,margin_streak_cum_lot,cash_streak_cum_lot,foreign_streak_cum_amt_thousand,it_streak_cum_amt_thousand,dealer_streak_cum_amt_thousand,inst_streak_cum_amt_thousand,margin_streak_cum_amt_thousand,cash_streak_cum_amt_thousand
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1101""",2026-07-30,1575.0,20.0,2003.0,879.0,-2882.0,4.0,3.0,1.0,4.0,1.0,-4.0,1.0504,-1.4344,-1.232,1.0504,-1.232,1.0504,38170.0,85.0,408.0,38973.0,879.0,-37514.0,930994.0,2072.0,9875.0,950483.0,21275.0,-914905.0
"""1102""",2026-07-30,4696.0,23.0,4881.0,-109.0,-4772.0,8.0,5.0,4.0,10.0,-1.0,-10.0,0.606,1.529,1.374,0.606,1.2195,0.606,17290.0,163.0,1186.0,31141.0,-109.0,-30761.0,570241.0,5348.0,38995.0,1.032492e6,-3606.0,-1.019748e6
"""1103""",2026-07-30,176.0,0.0,153.0,-3.0,-150.0,2.0,0.0,-1.0,2.0,-1.0,-2.0,-2.2305,-0.3788,-0.3788,-2.2305,-0.3788,-2.2305,340.0,0.0,-23.0,329.0,-3.0,-331.0,4502.0,0.0,-304.0,4358.0,-40.0,-4384.0
"""1104""",2026-07-30,627.0,0.0,617.0,-22.0,-595.0,3.0,0.0,-1.0,3.0,-1.0,-3.0,1.8726,1.3035,1.3035,1.8726,1.3035,1.8726,932.0,0.0,-10.0,932.0,-22.0,-910.0,25069.0,0.0,-271.0,25063.0,-595.0,-24468.0
"""1108""",2026-07-30,236.0,0.0,222.0,-3.0,-219.0,2.0,0.0,-1.0,2.0,-2.0,-2.0,-1.6598,-0.8368,-0.8368,-1.6598,-1.6598,-1.6598,329.0,0.0,-14.0,317.0,-11.0,-306.0,3943.0,0.0,-168.0,3799.0,-132.0,-3667.0


### institutional_net_flow — 三大法人買賣超


In [15]:
import institutional_net_flow

reload_modules([institutional_net_flow])
institutional_net_flow_df = institutional_net_flow.process_institutional_net_flow(
    read_tej_zip(
        zip_dir,
        "三大法人買賣超",
        institutional_net_flow.INSTITUTIONAL_NET_FLOW_SCHEMA,
    )
)
print("institutional_net_flow_df:", institutional_net_flow_df.shape)

institutional_net_flow_df = institutional_net_flow_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", institutional_net_flow_df.shape)


institutional_net_flow_df.head()

institutional_net_flow_df: (2332, 10)
after symbol filter: (1944, 10)


symbol,date,foreign_holding_pct,it_holding_pct,dealer_holding_pct,foreign_turnover_pct,it_turnover_pct,dealer_turnover_pct,dealer_prop_net_buy_thousand_shares,dealer_hedge_net_buy_thousand_shares
str,date,f64,f64,f64,f64,f64,f64,f64,f64
"""1101""",2026-07-30,14.53,0.15,0.07,0.2,0.0,0.01,401.0,7.0
"""1102""",2026-07-30,12.44,5.43,0.41,0.18,0.0,0.0,20.0,143.0
"""1103""",2026-07-30,4.54,0.0,0.02,0.03,0.0,0.0,-23.0,0.0
"""1104""",2026-07-30,14.84,0.02,0.03,0.1,0.0,0.0,-10.0,0.0
"""1108""",2026-07-30,1.27,0.0,0.03,0.04,0.0,0.0,-14.0,0.0


### Credit_Transactions — 融資融券


In [16]:
import Credit_Transactions

reload_modules([Credit_Transactions])
Credit_Transaction_df = Credit_Transactions.process_credit_transactions(
    read_tej_zip(
        zip_dir,
        "融資融券",
        Credit_Transactions.CREDIT_TRANSACTIONS_SCHEMA,
    )
)
print("Credit_Transaction_df:", Credit_Transaction_df.shape)

Credit_Transaction_df = Credit_Transaction_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", Credit_Transaction_df.shape)


Credit_Transaction_df.head()

margin: repay_ratio_na=118, balance_lot=16 (13.56%), balance_0=102 (86.44%)
short: repay_ratio_na=786, balance_lot=16 (2.04%), balance_0=770 (97.96%)
Credit_Transaction_df: (1867, 22)
after symbol filter: (1852, 22)


symbol,date,marginable,margin_usage_rate,short_usage_rate,short_to_margin_ratio,margin_change_ratio,short_change_ratio,margin_volume_pct,short_volume_pct,margin_maintenance_ratio,short_maintenance_ratio,account_maintenance_ratio,margin_buy_lot,margin_sell_lot,margin_cash_repay_lot,short_buy_lot,short_sell_lot,short_stock_repay_lot,short_last_cover_date,margin_cash_repay_ratio,short_stock_repay_ratio
str,date,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,f64,f64
"""1101""",2026-07-30,1,1.66,0.0,0.23,2.72,-44.53,2.98,0.11,165.26,191.65,165.36,1210.0,386.0,0.0,58.0,3.0,2.0,null,0.0,0.028169
"""1102""",2026-07-30,1,0.14,0.0,0.57,-7.78,250.0,0.53,0.02,163.74,188.62,163.97,7.0,111.0,0.0,0.0,5.0,0.0,null,0.0,0.0
"""1103""",2026-07-30,1,1.95,0.0,0.02,-0.07,0.0,0.66,0.0,155.67,192.46,155.68,2.0,5.0,0.0,0.0,0.0,0.0,null,0.0,0.0
"""1104""",2026-07-30,1,0.97,0.0,0.0,-1.31,0.0,2.33,0.0,158.07,null,158.07,16.0,38.0,0.0,0.0,0.0,0.0,null,0.0,null
"""1108""",2026-07-30,1,0.41,0.04,10.55,-0.71,0.0,0.7,0.0,144.29,198.82,151.5,1.0,4.0,0.0,0.0,0.0,0.0,null,0.0,0.0


### Day_Trading_Stats — 當日沖銷交易統計（當沖）


In [17]:
import parse
import Day_Trading_Stats

reload_modules([parse, Day_Trading_Stats])
Day_Trading_Stats_df = Day_Trading_Stats.process_day_trading_stats(
    read_tej_zip(
        zip_dir,
        "當日沖銷交易統計資訊",
        Day_Trading_Stats.DAY_TRADING_STATS_SCHEMA,
    )
)
print("Day_Trading_Stats_df:", Day_Trading_Stats_df.shape)

Day_Trading_Stats_df = Day_Trading_Stats_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", Day_Trading_Stats_df.shape)

Day_Trading_Stats_df.head()

Day_Trading_Stats_df: (1704, 5)
after symbol filter: (1691, 5)


symbol,date,day_trade_volume,day_trade_volume_mkt_pct,day_trade_margin_short_offset_pct
str,date,f64,f64,f64
"""1101""",2026-07-30,10599.0,39.6,0.0
"""1102""",2026-07-30,2320.0,20.77,0.0
"""1103""",2026-07-30,102.0,19.31,0.0
"""1104""",2026-07-30,147.0,12.69,0.0
"""1108""",2026-07-30,34.0,9.55,0.0


### Securities_Lending — 借券餘額
rebuild 標為有問題暫不用；每日 zip 有檔可載入，但 keep_col 只留 `symbol`/`date`，不併入 final。


In [18]:
import Securities_Lending

reload_modules([Securities_Lending])
Securities_Lending_df = Securities_Lending.process_securities_lending(
    read_tej_zip(
        zip_dir,
        "借券餘額",
        Securities_Lending.SECURITIES_LENDING_SCHEMA,
    )
)
print("Securities_Lending_df:", Securities_Lending_df.shape)


Securities_Lending_df = Securities_Lending_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", Securities_Lending_df.shape)



Securities_Lending_df.head()

Securities_Lending_df: (1724, 2)
after symbol filter: (1711, 2)


symbol,date
str,date
"""1101""",2026-07-30
"""1102""",2026-07-30
"""1103""",2026-07-30
"""1104""",2026-07-30
"""1108""",2026-07-30


### share_distribution — 籌碼分佈


In [19]:
import share_distribution

reload_modules([share_distribution])
share_distribution_df = share_distribution.process_share_distribution(
    read_tej_zip(
        zip_dir,
        "籌碼分佈(2008年起)",
        share_distribution.SHARE_DISTRIBUTION_SCHEMA,
    )
)
print("share_distribution_df:", share_distribution_df.shape)


share_distribution_df = share_distribution_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", share_distribution_df.shape)



share_distribution_df.head()

share_distribution_df: (2332, 16)
after symbol filter: (1944, 16)


symbol,date,foreign_trade_share_pct,it_trade_share_pct,dealer_trade_share_pct,credit_trade_share_pct,cash_trade_share_pct,free_float_turnover_pct,foreign_holding_ex_dir_pct,director_holding_pct,director_pledge_pct,margin_short_offset_pct,margin_increase_lot,margin_decrease_lot,short_increase_lot,short_decrease_lot
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1101""",2026-07-30,55.06,0.1,1.7,0.03,43.11,0.39,14.53,8.41,0.65,0.0,824.0,0.0,0.0,57.0
"""1102""",2026-07-30,56.84,0.43,0.94,0.01,41.78,0.43,12.44,26.19,2.14,0.0,0.0,104.0,5.0,0.0
"""1103""",2026-07-30,44.75,0.0,3.5,0.01,51.74,0.08,4.54,16.51,0.0,0.0,0.0,3.0,0.0,0.0
"""1104""",2026-07-30,60.85,0.0,0.95,0.02,38.18,0.18,14.84,4.58,0.0,0.0,0.0,22.0,0.0,0.0
"""1108""",2026-07-30,41.97,0.0,2.54,0.01,55.49,0.09,1.27,3.95,0.0,0.0,0.0,3.0,0.0,0.0


### Company_Info — 公司基本資料／產業（每日）
只保留 `symbol` / `TSE產業名`→`industry` / `TEJ子產業名`→`subindustry`。
再用 `setting_files/*_rebuild.json` 編成整數；若出現對照表沒有的產業／子產業會印出提示。


In [20]:
import company_info_daily
import Company_Info

reload_modules([company_info_daily, Company_Info])
setting_file = "/home/newt/quant_project/setting_files"

try:
    company_info_df = company_info_daily.process_company_info_daily(
        read_tej_zip(zip_dir, "基本資料", company_info_daily.COMPANY_INFO_DAILY_SCHEMA)
    )
    company_info_df, new_industries, new_subindustries = (
        company_info_daily.encode_daily_company_info(company_info_df, setting_file)
    )
    company_info_df = company_info_df.filter(
        pl.col("symbol").is_in(unique_symbol)
    )
    print("company_info_df:", company_info_df.shape, company_info_df.dtypes)
    if new_industries:
        print(f"⚠ 新 industry（不在對照表，已編成 null）: {new_industries}")
    if new_subindustries:
        print(f"⚠ 新 subindustry（不在對照表，已編成 null）: {new_subindustries}")
    if not new_industries and not new_subindustries:
        print("產業對照：無新增")
    display(company_info_df.head())
except FileNotFoundError:
    company_info_df = None
    print(f"skip company_info：{zip_dir} 無「基本資料」開頭的 zip")


company_info_df: (1944, 3) [String, Int16, Int16]
產業對照：無新增


symbol,industry,subindustry
str,i16,i16
"""1101""",0,0
"""1102""",0,0
"""1103""",0,0
"""1104""",0,0
"""1108""",0,0


## 載入摘要 — 各表 shape 一覽


In [22]:
_loaded = {
    "暫停交易": trading_halt_df,
    "日收盤價（未復權）": OHLCV_df,
    "現金股息": cash_dividend_df,
    "配股復權因子": ex_rights_adj_factor_df,
    "減資復權因子": capital_reduction_only_df,
    "減資停牌區間": capital_reduction_halt_df,
    "面額變更復權因子": par_value_change_only_df,
    "面額變更停牌區間": par_value_change_halt_df,
    "資本形成（原始）": ex_rights_df,
    "籌碼動向＆報酬": institutional_return_df,
    "三大法人買賣超": institutional_net_flow_df,
    "融資融券": Credit_Transaction_df,
    "當沖統計": Day_Trading_Stats_df,
    "借券餘額": Securities_Lending_df,
    "籌碼分佈": share_distribution_df,
    "公司基本資料": company_info_df,
}
for name, df in _loaded.items():
    if df is None:
        print(f"{name:20s}  SKIP")
    else:
        print(f"{name:20s}  {df.shape[0]:>8} rows × {df.shape[1]} cols")

print("有配股     :", has_stock_dividend)
print("有減資     :", has_capital_reduction)
print("有面額變更 :", has_par_value_change)

暫停交易                         1 rows × 4 cols
日收盤價（未復權）                 1944 rows × 25 cols
現金股息                        32 rows × 4 cols
配股復權因子                       3 rows × 4 cols
減資復權因子                       0 rows × 7 cols
減資停牌區間                       0 rows × 6 cols
面額變更復權因子                     0 rows × 7 cols
面額變更停牌區間                     1 rows × 6 cols
資本形成（原始）                    11 rows × 74 cols
籌碼動向＆報酬                   1944 rows × 31 cols
三大法人買賣超                   1944 rows × 10 cols
融資融券                      1852 rows × 22 cols
當沖統計                      1691 rows × 5 cols
借券餘額                      1711 rows × 2 cols
籌碼分佈                      1944 rows × 16 cols
公司基本資料                    1944 rows × 3 cols
有配股     : True
有減資     : False
有面額變更 : False


## 合併流程
現金除息先掛；配股／減資／面額與現金同日：參考價相同丟權利側，不同 raise。最後 `fill_margin_net_buy_lot_with_zero`。


In [23]:
import Credit_Transactions
reload_modules([Corporate_Actions, Credit_Transactions])


def _join_mul_adj(left: pl.DataFrame, right: pl.DataFrame) -> pl.DataFrame:
    """left join right 的 adj_factor：相乘後只留乘積（右為 null 當 1）。"""
    out = left.join(
        right.select(["symbol", "date", "adj_factor"]),
        on=["symbol", "date"],
        how="left",
    )
    return (
        out.with_columns(
            (pl.col("adj_factor") * pl.col("adj_factor_right").fill_null(1.0)).alias(
                "adj_factor"
            )
        ).drop("adj_factor_right")
    )


final_df = OHLCV_df.clone()

# 現金除息先掛；權利側再與現金做同日參考價重疊檢查（同價丟權利／異價 raise）
cash_for_overlap = cash_dividend_df if has_cash_dividend else None
if has_cash_dividend:
    final_df = _join_mul_adj(final_df, cash_dividend_df)

if has_stock_dividend:
    final_df = _join_mul_adj(
        final_df,
        Corporate_Actions.filter_rights_vs_cash_overlap(
            ex_rights_adj_factor_df, cash_for_overlap
        ),
    )

if has_capital_reduction:
    final_df = _join_mul_adj(
        final_df,
        Corporate_Actions.filter_rights_vs_cash_overlap(
            capital_reduction_only_df, cash_for_overlap
        ),
    )

if has_par_value_change:
    final_df = _join_mul_adj(
        final_df,
        Corporate_Actions.filter_rights_vs_cash_overlap(
            par_value_change_only_df, cash_for_overlap
        ),
    )

final_df = final_df.join(institutional_return_df, on=["symbol", "date"], how="left")
final_df = final_df.join(institutional_net_flow_df, on=["symbol", "date"], how="left")
final_df = final_df.join(Credit_Transaction_df, on=["symbol", "date"], how="left")
final_df = final_df.join(Day_Trading_Stats_df, on=["symbol", "date"], how="left")
# Securities_Lending 暫不進 DB（keep 只剩 symbol/date）；join 無新欄，略過
final_df = final_df.join(share_distribution_df, on=["symbol", "date"], how="left")
if company_info_df is not None:
    final_df = final_df.join(company_info_df, on=["symbol"], how="left")

final_df = Credit_Transactions.fill_margin_net_buy_lot_with_zero(final_df)


同日同參考價 2 筆 → 丟掉權利側、保留現金除息
shape: (2, 6)
┌────────┬────────────┬───────────┬──────────┬────────────┬──────────┐
│ symbol ┆ date       ┆ ref_price ┆ cash_ref ┆ adj_factor ┆ cash_adj │
│ ---    ┆ ---        ┆ ---       ┆ ---      ┆ ---        ┆ ---      │
│ str    ┆ date       ┆ f64       ┆ f64      ┆ f64        ┆ f64      │
╞════════╪════════════╪═══════════╪══════════╪════════════╪══════════╡
│ 1231   ┆ 2026-07-30 ┆ 80.72     ┆ 80.72    ┆ 1.118682   ┆ 1.118682 │
│ 1514   ┆ 2026-07-30 ┆ 94.8      ┆ 94.8     ┆ 1.041139   ┆ 1.041139 │
└────────┴────────────┴───────────┴──────────┴────────────┴──────────┘


In [24]:
import duckdb

duckdb_path = "/home/newt/quant_project/duckdb/daily_data.duckdb"
table_name = "daily_data"

conn = duckdb.connect(duckdb_path, read_only=True)
_desc = conn.execute(f"DESCRIBE {table_name}").fetchall()
db_df = pl.from_arrow(
    conn.execute(
        f"SELECT * FROM {table_name} WHERE date = ?::DATE",
        [check_date],
    ).fetch_arrow_table()
)
db_max_date = conn.execute(f"SELECT MAX(date) FROM {table_name}").fetchone()[0]
conn.close()

final_dates = sorted(final_df["date"].unique().to_list())
print(f"check_date : {check_date}")
print(f"final_df dates ({len(final_dates)}): {final_dates}")
print(f"DB 同日列數: {db_df.height} | final_df: {final_df.height}")
print(f"DB MAX(date): {db_max_date}")
if db_df.height == 0:
    raise ValueError(f"DuckDB 無 date={check_date} 的資料，無法對比")

db_cols = [r[0] for r in _desc]
db_types = {r[0]: r[1] for r in _desc}
final_cols = list(final_df.columns)

only_in_db = [c for c in db_cols if c not in set(final_cols)]
only_in_final = [c for c in final_cols if c not in set(db_cols)]
common = [c for c in db_cols if c in set(final_cols)]

print(f"\nDB {table_name}: {len(db_cols)} cols | final_df: {len(final_cols)} cols | 共同: {len(common)}")
print(f"final 缺少（DB 有）: {only_in_db if only_in_db else '無'}")
print(f"final 多出（DB 無）: {only_in_final if only_in_final else '無'}")


check_date : 2026-07-30
final_df dates (1): [datetime.date(2026, 7, 30)]
DB 同日列數: 1944 | final_df: 1944
DB MAX(date): 2026-07-30

DB daily_data: 101 cols | final_df: 101 cols | 共同: 101
final 缺少（DB 有）: 無
final 多出（DB 無）: 無


## 與 DuckDB 差異比對
以 `(symbol, date)` 對齊，比較共同欄位數值。
浮點用相對／絕對容差；null 對 null 視為相同。只顯示有差異的欄位與列。

In [25]:
KEYS = ["symbol", "date"]
cmp_cols = [c for c in common if c not in KEYS]
FLOAT_RTOL = 1e-9
FLOAT_ATOL = 1e-9

# --- key set 差異 ---
f_keys = set(zip(final_df["symbol"].to_list(), final_df["date"].to_list()))
d_keys = set(zip(db_df["symbol"].to_list(), db_df["date"].to_list()))
only_final = sorted(f_keys - d_keys)
only_db = sorted(d_keys - f_keys)
print(f"僅在 final（DB 無）: {len(only_final)}")
if only_final[:10]:
    print("  例:", only_final[:10])
print(f"僅在 DB（final 無）: {len(only_db)}")
if only_db[:10]:
    print("  例:", only_db[:10])

# --- 共同列逐欄比對 ---
left = final_df.select(KEYS + cmp_cols).rename({c: f"{c}__new" for c in cmp_cols})
right = db_df.select(KEYS + cmp_cols).rename({c: f"{c}__db" for c in cmp_cols})
joined = left.join(right, on=KEYS, how="inner")
print(f"\n共同 (symbol, date) 列數: {joined.height}")

diff_summary = []
diff_frames = []
for c in cmp_cols:
    a, b = f"{c}__new", f"{c}__db"
    dtype = joined[a].dtype
    both_null = pl.col(a).is_null() & pl.col(b).is_null()
    if dtype in (pl.Float32, pl.Float64):
        close = (pl.col(a) - pl.col(b)).abs() <= (
            FLOAT_ATOL + FLOAT_RTOL * pl.col(b).abs()
        )
        same = both_null | close.fill_null(False)
    else:
        same = both_null | (pl.col(a) == pl.col(b)).fill_null(False)
    n_diff = joined.filter(~same).height
    if n_diff:
        diff_summary.append((c, n_diff, str(dtype)))
        diff_frames.append(
            joined.filter(~same)
            .select(KEYS + [a, b])
            .with_columns(pl.lit(c).alias("column"))
        )

print(f"\n有差異欄位數: {len(diff_summary)} / {len(cmp_cols)}")
if not diff_summary:
    print("✓ 共同列、共同欄位完全一致（在浮點容差內）")
else:
    print(f"{'column':40s} {'n_diff':>8}  dtype")
    for c, n, dt in sorted(diff_summary, key=lambda x: -x[1]):
        print(f"{c:40s} {n:8d}  {dt}")
    all_diffs = pl.concat(diff_frames, how="diagonal_relaxed")
    print(f"\n差異列總覽（前 50）:")
    display(all_diffs.head(50))


僅在 final（DB 無）: 0
僅在 DB（final 無）: 0

共同 (symbol, date) 列數: 1944

有差異欄位數: 0 / 99
✓ 共同列、共同欄位完全一致（在浮點容差內）


In [27]:
final_df.filter(pl.col("symbol") == "5904").select('untradable').head()

untradable
i8
1


## 單欄差異細看（可選）
改 `inspect_col` 後重跑，查看該欄所有不一致列。

In [26]:
# 設成有差異的欄位名；空字串則跳過
inspect_col = "untradable"

if not inspect_col:
    print("設 inspect_col 後可細看該欄差異")
elif inspect_col not in cmp_cols:
    print(f"{inspect_col!r} 不在共同比對欄位：{cmp_cols}")
else:
    a, b = f"{inspect_col}__new", f"{inspect_col}__db"
    dtype = joined[a].dtype
    both_null = pl.col(a).is_null() & pl.col(b).is_null()
    if dtype in (pl.Float32, pl.Float64):
        close = (pl.col(a) - pl.col(b)).abs() <= (
            FLOAT_ATOL + FLOAT_RTOL * pl.col(b).abs()
        )
        same = both_null | close.fill_null(False)
    else:
        same = both_null | (pl.col(a) == pl.col(b)).fill_null(False)
    detail = joined.filter(~same).select(KEYS + [a, b]).sort("symbol")
    print(f"{inspect_col}: {detail.height} 列不同")
    display(detail)


untradable: 0 列不同


symbol,date,untradable__new,untradable__db
str,date,i8,i8


## 覆蓋寫入 DuckDB
只針對有欄位值差異（及 key 集合差）的 `symbol`：先 DELETE 同日該批，再 INSERT `final_df` 對應列（只寫 `db_cols`）。執行後互動確認。
若 `final` 缺少 DB 欄位則跳過；無差異則不寫入。


In [ ]:
if only_in_db:
    print(f"跳過寫入：final 缺少 DB 欄位 {only_in_db}")
else:
    write_dates = sorted(final_df["date"].unique().to_list())
    # 欄位值有差的 symbol；另含 key 集合差（僅 final 要補寫、僅 DB 要刪）
    value_diff_syms = (
        sorted(all_diffs["symbol"].unique().to_list()) if diff_summary else []
    )
    only_final_syms = sorted({s for s, _ in only_final})
    only_db_syms = sorted({s for s, _ in only_db})
    delete_syms = sorted(set(value_diff_syms) | set(only_final_syms) | set(only_db_syms))
    insert_syms = sorted(set(value_diff_syms) | set(only_final_syms))
    write_df = final_df.filter(pl.col("symbol").is_in(insert_syms)).select(db_cols)

    print(f"準備按 symbol 覆蓋寫入 {table_name}")
    print(f"  DB   : {duckdb_path}")
    print(f"  dates: {write_dates}")
    print(f"  欄位值差異 symbol: {len(value_diff_syms)} → {value_diff_syms[:20]}{'…' if len(value_diff_syms) > 20 else ''}")
    print(f"  僅 final（補寫）: {len(only_final_syms)} → {only_final_syms[:20]}{'…' if len(only_final_syms) > 20 else ''}")
    print(f"  僅 DB（刪除）  : {len(only_db_syms)} → {only_db_syms[:20]}{'…' if len(only_db_syms) > 20 else ''}")
    print(f"  DELETE symbols: {len(delete_syms)} | INSERT rows: {write_df.height}")

    if not delete_syms and write_df.height == 0:
        print("無差異，不寫入")
    else:
        ans = input("確認只覆寫上述 symbol？[y/N] ").strip().lower()
        if ans not in ("y", "yes"):
            print("已取消，未寫入")
        else:
            con = duckdb.connect(duckdb_path, read_only=False)
            n_before = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
            n_del = 0
            if delete_syms:
                n_del = con.execute(
                    f"""
                    SELECT COUNT(*) FROM {table_name}
                    WHERE date IN (SELECT UNNEST(?::DATE[]))
                      AND symbol IN (SELECT UNNEST(?::VARCHAR[]))
                    """,
                    [write_dates, delete_syms],
                ).fetchone()[0]
                con.execute(
                    f"""
                    DELETE FROM {table_name}
                    WHERE date IN (SELECT UNNEST(?::DATE[]))
                      AND symbol IN (SELECT UNNEST(?::VARCHAR[]))
                    """,
                    [write_dates, delete_syms],
                )
            if write_df.height:
                con.register("_daily_replace", write_df)
                con.execute(
                    f"INSERT INTO {table_name} BY NAME SELECT * FROM _daily_replace"
                )
            n_after = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
            max_date = con.execute(f"SELECT MAX(date) FROM {table_name}").fetchone()[0]
            con.close()
            print(
                f"已刪除同日指定 symbol {n_del} 列，寫入 {write_df.height} 列"
                f"（dates={write_dates}, symbols={len(delete_syms)}）"
            )
            print(f"表總列數: {n_before} → {n_after} | MAX(date): {max_date}")
